# 03 — MediaPipe landmark extraction

Extract up to two hands per static RGB image, rotation-invariant hand-shape descriptors, wrist position/scale, and body-relative hand geometry from MediaPipe Pose. Left and right hands occupy fixed slots; absent hands are zero alongside an explicit presence mask. Features remain keyed to each image's relative path and include label/hash metadata.

**Review the visual sample output before enabling cache extraction.** This ordering helps catch detector or handedness problems before generating the full cache.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import load_config, project_path
from src.dataset import load_split_manifest
from src.landmark_extractor import HAND_FEATURE_DIM, LANDMARK_CACHE_FORMAT_VERSION, LANDMARK_FEATURE_DIM, HandLandmarkExtractor, extract_landmark_cache

config = load_config()
manifest_path = project_path(config, "splits") / "split_manifest.json"
cache_path = project_path(config, "landmarks") / "mediapipe_landmarks.json"
if not manifest_path.is_file():
    manifest = None
    records = []
    print("Split manifest is missing. Run 02_data_preprocessing.ipynb on the real dataset first.")
else:
    manifest = load_split_manifest(manifest_path)
    record_map = {row["relative_path"]: row for rows in manifest["splits"].values() for row in rows}
    records = list(record_map.values())
    if not records:
        raise ValueError("Split manifest contains no image records.")
    print(f"Manifest images: {len(records)} | Cache target (not written yet): {cache_path}")

## Visual detector quality check — run before extraction

In [ ]:
if manifest is not None:
    import numpy as np
    import matplotlib.pyplot as plt
    from PIL import Image

    examples_by_class = {}
    for record in records:
        examples_by_class.setdefault(record["label"], record)
    examples = [examples_by_class[label] for label in sorted(examples_by_class)[:8]]
    preview_results = []
    images_are_mirrored = bool(config["data"].get("images_are_mirrored", False))
    with HandLandmarkExtractor(input_is_mirrored=images_are_mirrored) as extractor:
        figure, axes = plt.subplots(len(examples), 2, figsize=(9, 3 * len(examples)), squeeze=False)
        for row_index, record in enumerate(examples):
            with Image.open(record["path"]) as image:
                rgb = np.asarray(image.convert("RGB"))
            detection = extractor.extract(rgb)
            preview_results.append((record, detection))
            axes[row_index, 0].imshow(rgb)
            axes[row_index, 0].set_title(f"{record['label']} — source")
            axes[row_index, 1].imshow(extractor.draw(rgb, detection))
            axes[row_index, 1].set_title(f"presence={detection['presence'].astype(int).tolist()}, body={int(detection['features'][-1])}")
            for axis in axes[row_index]:
                axis.axis("off")
        plt.tight_layout()
        plt.show()
    preview_detected_rate = sum(result["detected"] for _, result in preview_results) / max(len(preview_results), 1)
    print(f"Preview images with at least one hand: {preview_detected_rate:.1%}")
    preview_body_rate = sum(result['features'][-1] > 0.5 for _, result in preview_results) / max(len(preview_results), 1)
    print(f"Preview images with visible nose and shoulders (body-relative features available): {preview_body_rate:.1%}")
    print("Inspect the images above, then confirm the review in the next code cell before extracting the full cache.")
else:
    preview_results = []
    LANDMARKS_VISUALLY_VALIDATED = False
    print("Visual validation skipped until preprocessing has produced a split manifest.")

In [ ]:
# Review the sample detections above, then change this to True and run this cell.
LANDMARKS_VISUALLY_VALIDATED = False
print("Visual validation confirmed:", LANDMARKS_VISUALLY_VALIDATED)

## Extract/cache and validate aligned features

The cache is written only after the explicit visual-review flag is enabled. If a matching cache already exists, image keys and SHA-256 values are checked before reuse; stale caches are recomputed. MediaPipe failures are raised, not converted into success-shaped empty data.

In [ ]:
if manifest is None:
    print("Extraction skipped: the split manifest is unavailable.")
elif not LANDMARKS_VISUALLY_VALIDATED:
    print("Extraction not started. Review the visual output, set LANDMARKS_VISUALLY_VALIDATED = True in the visual-check cell above, rerun it, then rerun this cell.")
else:
    import json
    import numpy as np
    import pandas as pd

    expected = {record["relative_path"]: record for record in records}
    cache = None
    if cache_path.is_file():
        existing = json.loads(cache_path.read_text(encoding="utf-8"))
        items = existing.get("items", {})
        if existing.get("format_version") == LANDMARK_CACHE_FORMAT_VERSION and existing.get("feature_dim") == LANDMARK_FEATURE_DIM and existing.get("input_is_mirrored", False) == bool(config["data"].get("images_are_mirrored", False)) and set(items) == set(expected) and all(items[key].get("sha256") == expected[key]["sha256"] for key in expected):
            cache = existing
            print("Reusing cache with matching paths and hashes.")
        else:
            print("Existing cache is stale or incomplete; recomputing it.")
    if cache is None:
        cache = extract_landmark_cache(records, cache_path, input_is_mirrored=bool(config["data"].get("images_are_mirrored", False)))
    items = cache["items"]
    assert set(items) == set(expected), "Landmark cache and split manifest image keys differ"
    for relative_path, record in expected.items():
        item = items[relative_path]
        assert item["label_idx"] == record["label_idx"] and item["label"] == record["label"]
        assert item["sha256"] == record["sha256"]
        assert len(item["features"]) == LANDMARK_FEATURE_DIM and len(item["positions"]) == 8 and len(item["presence"]) == 2 and len(item["handedness_reliable"]) == 2
    presence_matrix = np.asarray([item["presence"] for item in items.values()])
    print(f"Cache alignment validated for {len(items)} images: {cache_path}")
    print(f"Images with >=1 detected hand: {(presence_matrix.sum(axis=1) > 0).mean():.1%}")
    print(f"Images with two detected hands: {(presence_matrix.sum(axis=1) == 2).mean():.1%}")
    print(f"Images with body-relative features: {np.mean([item['features'][-1] > 0.5 for item in items.values()]):.1%}")
    coverage_rows = [
        {"split": split_name, "class": record["label"], "body_valid": items[record["relative_path"]]["features"][-1] > 0.5}
        for split_name, split_rows in manifest["splits"].items()
        for record in split_rows
    ]
    coverage = pd.DataFrame(coverage_rows).groupby(["class", "split"])["body_valid"].mean().unstack("split")
    print("\nNose-and-shoulder coverage by class and split (%):")
    print((coverage * 100).round(1).to_string())